In [1]:
# Scientific computing.
import pandas as pd
import numpy as np
from collections import Counter

# Machine learning.
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    confusion_matrix,
)

# File handling.
from pathlib import Path
import json

In [2]:
# Anchor to the notebook's location to not hardcode paths.
Notebook_Dir = Path().resolve()
Project_Root = Notebook_Dir.parents[1]
Data_Dir = Project_Root / "Clean_Data_Resources"

# Load the parquet from Text_Parsing.ipynb.
Survey_df = pd.read_parquet(Data_Dir / "Survey_df_Text_Parsed.parquet")

# Load scale guide that maps each column/semester/discipline to its scale type.
# From Organization.ipynb.
Likert_Guide_df = pd.read_csv(Data_Dir / "Survey_Results_Likert_Guide.csv")

# Load column metadata.
# From Organization.ipynb
with open(Data_Dir / "column_metadata.json", "r") as f:
    Column_Metadata = json.load(f)

In [3]:
# Paste in Likert scales, for sanity.
# There are two different likerts that are used, where the interpretations are slightly different. 
agreement_scale = {
    'Strongly agree': 5,
    'Moderately agree': 4,
    'Neither disagree nor agree': 3,
    'Moderately disagree': 2,
    'Strongly disagree': 1,
    'Unable to judge': None
}
# Agreement with statements is not the same as assessing helpfulness directly. 
# "Moderately helpful" is not the same as "neither disagree nor agree".
helpfulness_scale = {
    'Extremely helpful': 5,
    'Very helpful': 4,
    'Moderately helpful': 3,
    'Slightly helpful': 2,
    'Not at all helpful': 1,
    'Unable to judge': None
}
# Combine both mappings in a dictionary.
rating_encoding = {**agreement_scale, **helpfulness_scale}

## Step one: Primary Pairs.
T_ Columns are paired with matching R_ columns, based on theme. 
 For instance: The open response column with understanding collaboration is paired with the equivalent rating column that also measures collaborative activities. 

In [4]:
# Leader rating columns for averaging.
Leader_R_Cols = [
    "R_Leader_Helps_Understanding_encoded",
    "R_Leader_Subject_Competence_encoded",
    "R_Leader_Has_Plan_encoded",
    "R_Leader_Willing_To_Help_encoded",
]

# T_ column to its matched R_ column.
Primary_Pairs = [
    ("T_Collaboration_Understanding",      "R_Collaborative_Activities_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Helps_Understanding_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Subject_Competence_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Has_Plan_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Willing_To_Help_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Average_encoded"),
    ("T_Other_Suggestions",                "R_Overall_Session_Helpfulness_encoded"),
    ("T_Program_Overall_Suggestions",      "R_Overall_Session_Helpfulness_encoded"),
]

In [5]:
# Create an average leader score metric.and
# T_Leader_Performance_Suggestions corresponds with columns that rate leader performance.
Survey_df["R_Leader_Average_encoded"] = Survey_df[Leader_R_Cols].mean(axis=1)

In [6]:
# Build Agreement-scale row index per R_ column from Likert_Guide_df.
# Agreement Likerts are the vast majority of responses. Helpfulness Likerts aren't.

def get_agreement_index(r_col):
    # Strip _encoded suffix to match Likert_Guide_df column names.
    col_name = r_col.replace("_encoded", "")
    if col_name == "R_Leader_Average":
        # For averaged column, intersect Agreement rows across all four leader columns.
        indices = None
        for leader_col in Leader_R_Cols:
            agreement_rows = Likert_Guide_df[
                (Likert_Guide_df["Column"] == leader_col.replace("_encoded", "")) &
                (Likert_Guide_df["Scale"] == "Agreement")
            ][["Discipline", "Course_Code", "Semester", "Year"]]
            mask = Survey_df[["Discipline", "Course_Code", "Semester", "Year"]].merge(
                agreement_rows, how="inner"
            ).index
            indices = mask if indices is None else indices.intersection(mask)
        return indices
    # For single columns, filter directly.
    agreement_rows = Likert_Guide_df[
        (Likert_Guide_df["Column"] == col_name) &
        (Likert_Guide_df["Scale"] == "Agreement")
    ][["Discipline", "Course_Code", "Semester", "Year"]]
    mask = Survey_df.merge(
        agreement_rows,
        on=["Discipline", "Course_Code", "Semester", "Year"],
        how="inner"
    ).index
    return mask

In [7]:
# Bag of words builder.
def build_bow(token_lists):
    # Build vocabulary from all token lists.
    vocab = sorted(set(tok for tokens in token_lists for tok in tokens))
    vocab_index = {tok: i for i, tok in enumerate(vocab)}
    # Build count matrix.
    matrix = np.zeros((len(token_lists), len(vocab)), dtype=int)
    for i, tokens in enumerate(token_lists):
        counts = Counter(tokens)
        for tok, count in counts.items():
            if tok in vocab_index:
                matrix[i, vocab_index[tok]] = count
    return matrix, vocab

In [8]:
def get_top_features(nb_model, vocab, n=20):
    # Extract top n tokens most predictive of each class.
    top_features = {}
    for i, class_label in enumerate(nb_model.classes_):
        top_indices = nb_model.feature_log_prob_[i].argsort()[-n:][::-1]
        top_features[class_label] = [vocab[j] for j in top_indices]
    return top_features

In [9]:
def prepare_model_data(t_col, r_col):
    # Get Agreement-scale row indices for this pairing.
    agreement_idx = get_agreement_index(r_col)
    subset = Survey_df.loc[agreement_idx].copy()
    # Combine lemmas and bigrams as features.
    subset["features"] = (
        subset[t_col + "_lemmas"] + subset[t_col + "_bigrams"]
    )
    # Drop rows with no text, no rating, or neutral rating.
    # We throw out threes. Why? They mean "neither agree nor disagree."
    subset = subset.dropna(subset=[r_col])
    subset = subset[subset["features"].apply(len) > 0]
    subset = subset[subset[r_col] != 3]
    # Binarize rating: above 3 is 1, below 3 is 0 (negative sentiment).
    y = (subset[r_col] > 3).astype(int)
    # Check we have enough data and both classes present.
    if len(y) < 50 or y.nunique() < 2:
        return None, None, None
    return subset, y, subset["features"].tolist()

In [11]:
def split_and_train(features, y):
    # Build BOW from combined lemmas and bigrams.
    X, vocab = build_bow(features)

    # 80/20 stratified split.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Compute vocabulary coverage — tokens seen in training that appear in test.
    train_vocab = set(
        tok for tokens in features[:len(y_train)] for tok in tokens
    )
    test_tokens = set(
        tok for tokens in features[len(y_train):] for tok in tokens
    )
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0
    
    # Train model.
    nb = MultinomialNB()
    nb.fit(X_train, y_train)
    return nb, vocab, X_test, y_train, y_test, vocab_coverage